# Repository Discovery

It is difficult, without specialized knowledge or dedicated inspection, to determine how repos on Github are related to one another.

Which projects (repos) are trying to solve the same problem? Which are in the same ecosystem? Which pre-date which others?

We've begun to study this problem using the abstract model of networks. Our approach is summarized by the neuroscience Hebbian mneumonic approximation: "fire together, wire together."
Said with greater context, groups of projects that share features (contributors, in this first exploration) are more likely than others to do similar things.

We expect projects in the container orchestration space to, pairwise, share more contributors together than with projects outside of that context, on average.

## This notebook's contribution

In this notebook, we attempt to solve a challenging problem:

**Given the contributors in a known kernel of projects within the same domain, identify relevant, previously censored, repos that those contributors have also taken part in.**

Similarly,

**Given a list of newly discovered potential ecosystem member repositories, identify those that are the most preeminent, filtering out those that are natural 'noise,' irrelevant to the studied ecosystem.**

# Setup

Import Python packages, load database access credentials, define connection to DB.

In [ ]:
#IMPORTS
import pandas as pd
import sqlalchemy as salc
import plotly.express as px
import json
import numpy as np
from IPython.display import Image
import math

In [ ]:
#DATABASE ACCESS

with open("wasm_creds.json") as config_file:
    config = json.load(config_file)

database_connection_string = f"postgresql+psycopg2://{config['user']}:{config['password']}@{config['host']}:{config['port']}/{config['database']}"

engine = salc.create_engine(
    database_connection_string,
    connect_args={'options': f"-csearch_path={'augur_data'}"})

In [ ]:
# SWITCHES

LOCAL_DATA = True 

# Known contributors

At the outset, we identified a kernel of projects that we are highly confident exist within our ecosystem of interest.

**NOTE:** We refer to 'kernel' here as an abstract mathematical centerpoint of our analysis. We aren't referring to the software artifact at the center of an operating system.

From this kernel of projects, we identified all of the contributors and ranked them based on how they interacted with one another.

Those who interact with many others in a valuable way are more highly ranked than those who never interact whatsover, and soforth.

The data below is the artifact from that analysis, where each row is: {contributor_id, contributor_importance_ranking}

---

For reference, this work is at the following absolute path in this repository:

`Rappel/notebooks/8knot/collab_network/wasm/collabs.ipynb`


-- 

Small overview of pagerank ~

Pagerank is an algorithm for measuring the importance of nodes in a network based on their connectivity to other nodes. It's slightly difference from betweenness-centrality or closeness-centrality metrics. 

In order to get a pagerank score for each contributor, we build a social network based on interactions between contributors on github. Then, we run the pagerank algorithm on this network, yielding a scoring of contributors, implying their social 'importance' within the greater social group. People with higher pagerank scores can be interpreted as more densely connected, communicative, and involved than those with lower scores, although this is a generalization.

Contributor's pagerank scores won't be used directly in this analysis, but will in the future.

In [ ]:
df_known_contribs = pd.read_csv('contrib_pagerank_scores.csv')
df_known_contribs = df_known_contribs.rename(columns={"contrib_id": "cntrb_id"})
df_known_contribs.head()

# Contribution event stream

Github provides an API endpoint from which one can query every event that occurs. For instance, if a contributor creates an issue on some repo, that event will be logged in the stream. A downside of this API is that it only serves the most recent two months of data. 

We want to use this stream to identify the set of repositories that our Known Contributors are also working in that weren't in the initial ecosystem kernel.

The database that we use collects this event stream so we can query it from there rather than from Github. This is much faster than relying on the public Github API, and we will continue to collect the data that the API makes available, so over time the analysis that we can do will become richer and more complete. In the Augur database, the table with this information is the 'contributor_repo' table.

In [ ]:
event_stream_query = salc.sql.text(
    f"""
        SET SCHEMA 'augur_data';
        SELECT 
            c.cntrb_id,
            c.event_id,
            c.created_at,
            c.cntrb_repo_id as repo_id,
            c.repo_git,
            c.repo_name,
            c.gh_repo_id,
            c.cntrb_category as event_type
        FROM
        contributor_repo c
    """)


if not LOCAL_DATA:
    with engine.connect() as conn:
        df_event_stream = pd.read_sql_query(event_stream_query, conn)
        
    with open("df_event_stream.parquet", "wb+") as f:
        df_event_stream.cntrb_id = df_event_stream.cntrb_id.astype(str)
        df_event_stream.to_parquet(f)
else:
    with open("df_event_stream.parquet", "rb+") as f:
        df_event_stream = pd.read_parquet(f)

## Event stream data

Below we see a summary of the data we have:

- 4.2 million events,

In [ ]:
df_event_stream.describe()

### Head of dataframe

In [ ]:
df_event_stream.head()

### Most common types of events

In [ ]:
event_counts = df_event_stream.event_type.value_counts()
event_counts

In [ ]:
fig_ec = px.bar(
    data_frame=event_counts.to_frame(),
    color=event_counts.to_frame().index,
    labels={
                     "value": "# Events",
                     "event_type": "Kind of Event"
                 },
)
fig_ec.update_layout(showlegend=False)
fig_ec.write_image("general_eventcount.png")
fig_ec

In [ ]:
Image("general_eventcount.png")

### Repositories with the most events 

In [ ]:
repo_event_counts = df_event_stream.repo_git.value_counts()
repo_event_counts

In [ ]:
df_fig_rec = repo_event_counts.to_frame().rename(columns={"repo_git": "count"})
print(df_fig_rec)
df_fig_rec = df_fig_rec[df_fig_rec["count"] > 2500]

fig_rec = px.bar(
    data_frame=df_fig_rec,
    color=df_fig_rec.index,
)
fig_rec.update_layout(showlegend=False)
fig_rec.write_image("general_repository_eventcount.png")
fig_rec

In [ ]:
Image("general_repository_eventcount.png")

In [ ]:
df_fig_rec = repo_event_counts.to_frame().rename(columns={"repo_git": "count"})
print(df_fig_rec)
df_fig_rec = df_fig_rec[df_fig_rec["count"] > 2500]

fig_rec = px.bar(
    data_frame=df_fig_rec,
    color=df_fig_rec.index,
    labels={
                    "value": "# Contributor Events",
                    "repo_git": "Repository URL"
                },
)
fig_rec.update_layout(showlegend=False)
fig_rec.write_image("general_repository_eventcount.jpeg")
fig_rec

In [ ]:
Image("general_repository_eventcount.jpeg")

## Contribution event stream re: Known contributors

We'd like to know which projects our group of contributors are the most active in. 

We'll cross-reference our event stream with the list of contributor IDs we have from our previous analysis.

In [ ]:
UUID_known = df_known_contribs.cntrb_id.to_list()
UUID_known[:10]

In [ ]:
# only consider events from known contributors.
df_known_event_stream = df_event_stream[df_event_stream.cntrb_id.isin(UUID_known)]

### Difference in size:

The initial set of events represented the activity of the population of all Github users. 

By filtering by UUID_known, we now consider only those events from those contributors that we identified in our previous step; those that were in the initial set of repos that we believe are in our ecosystem of interest.

This change is from 4.2 Million events to 454K events, nearly an order of magnitude difference.

In [ ]:
print(f"Diff in size: {df_event_stream.shape} compared to {df_known_event_stream.shape}")

## Most common events among known contributors

In [ ]:
kc_event_counts = df_known_event_stream.event_type.value_counts()
fig_kec = px.bar(
    data_frame=kc_event_counts.to_frame(),
    color=kc_event_counts.to_frame().index,
    labels={
                     "value": "# Events",
                     "event_type": "Kind of Event"
                 },
)
fig_kec.update_layout(showlegend=False)
fig_kec.write_image("knowncontributors_eventcount.png")
fig_kec

In [ ]:
Image("knowncontributors_eventcount.png")

## Most common repositories among known contributors

In [ ]:
kc_repo_event_counts = df_known_event_stream.repo_git.value_counts()
df_fig_rec_kc = kc_repo_event_counts.to_frame() 

fig_rec_kc = px.bar(
    data_frame=df_fig_rec_kc[:50],
    color=df_fig_rec_kc[:50].index,
    labels={
                    "value": "# Contributor Events",
                    "repo_git": "Repository URL"
                },
)
fig_rec_kc.update_layout(showlegend=False)
fig_rec_kc.write_image("knowncontributors_repository_eventcount.png")
fig_rec_kc

In [ ]:
Image("knowncontributors_repository_eventcount.png")

## Interpretation

We find here that some obvious WASM-specific repositories are contributed to most frequently by the contributors we already knew about.

However, as one might expect, repos that are popular among the general public like 'risingwavelabs/risingwave', (which is 5th in this list and 74th among the general population, despite being a distributed streaming DB (not about WASM))
are also popular among this slice of the general population.

Our challenge now is to find those repos that are *especially* popular among this population as compared to the popularity among the general public.

# Identify disproportionately popular repos

## Unknown contributors

We want to consider the popularity of repos among people who aren't in our initial set.

If our 'known' group of contributors is the subpopulation "p" and the general population is "P",
then the group of people who are in the general population but aren't in our sub-population is "P - p" or " P^ ", pronounced P-hat.

Hence, P-hat would be our unknown contributors.

In [ ]:
# Get events made by those in P^

df_general_event_stream = df_event_stream[~df_event_stream.cntrb_id.isin(UUID_known)]
df_general_event_stream.shape

In [ ]:
# Count the events-per-repo of P^

g_repo_event_counts = df_general_event_stream.repo_git.value_counts().to_frame()
g_repo_event_counts

In [ ]:
# Join the counts of P^ with those of p

joined_counts = g_repo_event_counts.join(
    other=kc_repo_event_counts.to_frame(),
    how="left",
    lsuffix="_general",
    rsuffix="_known",
)

joined_counts = joined_counts.rename(columns={"repo_git_general": "count_general", "repo_git_known": "count_known"})
joined_counts = joined_counts.fillna(0)
joined_counts

## Simple approach

In this figure we visualize the count of contributions from those in 'p' and 'P^' stacked on top of one another. 

Most of the most-contributed-to projects of those in 'p' are also most-contributed-to by members of 'p' (they are not also largely contributed to by those in 'P^').

Some notable visual exceptions are 'NixOS/nixpkgs,' 'dotnet/runtime,' 'astral-sh/ruff,' 'paritytech/substrate,' and 'llvm/llvm-project.'

In [ ]:
fig_joined_counts = px.bar(
    data_frame=joined_counts
    .sort_values(by="count_known", ascending=False) # sort the joined_counts df by the number of contributions made by those in p
    [:50], # only take the first 50
    y=["count_general", "count_known"],
    labels={
                    "value": "# Contributor Events",
                    "repo_git": "Repository URL"
                },
)
fig_joined_counts.update_layout(showlegend=False)
fig_joined_counts.write_image("joined_counts.png")
fig_joined_counts

In [ ]:
Image("joined_counts.png")

## Interpretation

Above, the Red parts of the bars represent the count of events from our known population 'p', whereas the Blue parts of the stacks represent the complementary population 'P^'

From left to right, descending, are the projects with the largest number of contribution events (number of contribution events) from people in 'p'.

The people from our subpopulation should 100% dominate projects like Wasmer and Wasmedge, because those are among the populations whose population we're directly considering.
(wasmedge, wavm, wasmtime, wasmr are the repos we're pulling from). Interestingly though, we also see that the same group of people dominate other, non-WASM projects like risingwave, noir, and mathlib4.

This opens an interesting new line of inquiry. If the overlap of contributors between projects was extremely ecosystem-specific, e.g. people from WASMedge MOSTLY participated in WASM related projects,
then we could easily start with any WASM project and find those projects with greatest contributor overlap, and identify that those are all the WASM projects. 

However, this isn't the case- people who are participating in WASM are also participating in non-WASM projects, and they're contributing in high volumes, 'dominating' those projects w.r.t the population 'P^'.

## Conclusion

There isn't free lunch here. We need to use more sensitive methods to 'sniff out' WASM-related repositories, considering not only WHO is contributing WHERE, but who their social network is and
what their behavior is within the 'seed' communities we start with.


## Implement alpha and beta indices of preference

Mathematical definition of following approach:

https://www.overleaf.com/read/hxnsqsydqcrw#f624f4

### Implement alpha

'alpha' is a probablistic contribution metric. It captures 'how much more or less likely' the subpopulation 'p' is to contribute to a repository relative to the general population 'P^'.

More details can be found in the overleaf document referenced at the top of this section.

In [ ]:
# probabilities of contribution in known population

# number of contribution events by our contributor subpopulation
n_known = joined_counts["count_known"].sum()

# divide the per-repo contribution count by the total num of contributions by population.
# this is interpretateable as "the probability that the subpopulation will contribute to a given repo"
joined_counts["p_known"] = joined_counts["count_known"] / n_known
joined_counts

In [ ]:
# Probabilities of contribution in general population:

# number of contribution events by the general population.
n_general = joined_counts["count_general"].sum()

# divide the per-repo contribution count by the total num of contributions by general population.
# this is interpretateable as "the probability that the general population will contribute to a given repo"
joined_counts["p_everyone"] = (joined_counts["count_general"] + joined_counts["count_known"]) / (n_general + n_known)
joined_counts

In [ ]:
# calculate alpha

# by what factor is the subpopulation more or less likely to contribute to a repo?
joined_counts["alpha"] = joined_counts["p_known"] / joined_counts["p_everyone"]
joined_counts

In [ ]:
joined_counts.sort_values(by=["alpha"], ascending=False)

Those repos that our subcommunity dominates (those that they own) are ~17x more likely to be contributed to (naturally) than by the general population.

e.g. for 'moondance-labs/tanssi', |p|=683 and |P^|=1, so the probability that the subpopulation contributes to this particular project is measured as significantly higher than the general population.

However, 683 events still isn't many relative to the number of events we observe for large projects (9443 for apple/swift), so this isn't a terribly strong indication that the project is 'important' to the WASM ecosystem in general.

In [ ]:
# visualize alpha scores

fig = px.bar(
    data_frame = joined_counts.sort_values(by="alpha", ascending=False)[:100],
    y="alpha",
    custom_data=["count_known", "count_general"]
)
fig.update_traces(hovertemplate = "Repo:%{label}: <br>Alpha: %{value} </br>(count_known, count_general) : %{customdata}"
)
fig.write_image("alpha_scores.png")
fig

In [ ]:
Image("alpha_scores.png")

To understand this graph, let's take a given alpha value and interpret it.

For moondance-labs/tanssi, the alpha value is 16.93 - we can interpret this value as "Members of the subpopulation 'p' are 16.9x more likely to contribute to this repository than members of the general population 'P^' are."

Most of the alpha values we see in this top-100 alpha values plot are similar enough that the graph looks flat, but they're all slightly different values, based on count_general and count_known.

Observing the top 1000 repos by alpha value, the alpha value decreases fairly steadily without major outliers.

### Implement beta

'beta' attempts to connect the probablistic metric that 'alpha' provides to the real volume of contribution events that a repo experiences. The number of events is log-scaled so that orders-of-magnitude differences are smoothed out.

Ideally, 'beta' allows us to see which repos are not only disproportionately 'popular' among the subpopulation (which can be biased toward projects that are ONLY contributed to by our subpopulation) but are also GENERALLY seeing lots of work being done.

In [ ]:
# take the log of the count of known contributors.
# This reduces the impact of very high contribution volume, shifting the 
# impact toward the alpha value.

joined_counts["log_count_known"] = joined_counts["count_known"].apply(lambda x: math.log(x) if x>0 else 0)

In [ ]:
joined_counts["beta"] = joined_counts["alpha"] * joined_counts["log_count_known"] 
joined_counts[:10]

In [ ]:
fig = px.bar(
    data_frame=joined_counts.sort_values(by="beta", ascending=False)[:100],
    y="beta",
    hover_data=["alpha", "log_count_known", "count_known", "count_general"]
)
fig.write_image("beta_scores.png")
fig

In [ ]:
Image("beta_scores.png")

'beta' as a raw number is fairly arbitrary, but as a relative value, it lends to an informative comparison between projects.

'Higher' beta values "score better" w.r.t popularity and contribution volume from contributors, while 'lower' values are likely lower in contribution volume or in proportional popularity than their peer projects.
The balance between the impact of these two considerations deserves significant refinement, and is an opportunity for further work.

Nevertheless, we can see above that "WASM-y" projects have the highest 'beta' values for our subpopulation of interest, including some that we didn't have in our 'kernel' group of projects.

## Takeaway

Top repos by 'beta' value are:

- risingwave
- wasmer
- cosmwasm
- wasmtime
- ppsspp
- tigerbeetle
- ion-rust
- redash
- Ambient
- rspack
- socket
- runwasi
- holochain

Many of these are very directly connected to WASM, while others (risingwave, tigerbeetle, redash, holochain, ppsspp) are relatively unrelated.

This demonstrates that simply considering the contribution-base of our sub-population will yield strong cross-pollination signal. We need to be considerate of ecosystem-specific features when selecting our contributor set in order to clarify the ecosystem discovery signal.

### Next step

Top repos by 'beta' are similar to those by 'alpha.' 

The distribution of projects seems to have some scew toward database systems and no-trust mathematical proofs. This probably represents the interest-space of contributors within the WASM space, which itself is fairly experimental.

The next logical step is to renew focus on the 'core' contributors of the collaboration network, weighing the contributions of those people highly because they're highly integrated in the world of WASM.

# Conclusion

The alpha and beta metrics proposed in this work attempt to inspect the distinction between projects that are 'popular' within a subgroup working on an ecosystem of interest
and those that are 'massively contributed to.' A project might be very popular numerically within a subpopulation because they're the exclusive contributors to that project,
but the amount of work being done might be comparatively low. Likewise, a project could be astonomically interesting to the general population, and thereby interesting to many people
within our subpopulation, but be RELATIVELY less interesting compared to other projects that our subpopulation is working on.

This notebook applies alpha and beta to the behavior of our subgroup and identifies projects that are both popular and largely contributed to by our subpopulation of interest.

Notably, we can see that the "most popular" general project, LLVM, is not present in the 'top repos by beta value', but projects that are very WASM-y and previously unknown are included,
such as cosmwasm and runwasi.

## Further Work

These results are interesting but they further substantiate that count-based identification of repositories within an ecosystem is a weak signal at best.
Future work should look beyond simply "counting people" and "counting contribution events."